# Phase 9: Cooling Recommendation

This notebook loads persisted Phase 9 decision-support recommendations over frozen Phase 8 risk outputs. It does not retrain models or claim a certified controller.

## Frozen contract

The forecasting models remain Extended Ridge(alpha=1.0) at 60s and Extended OLS at 300s. Phase 8 risk categories remain frozen. Phase 9 derives only recommendation rules from validation runs ID16 and ID22.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
results_dir = project_root / 'results' / 'freetwinev'
figures_dir = results_dir / 'figures'
config = json.loads((results_dir / 'phase9_recommendation_config.json').read_text(encoding='utf-8'))
validation = pd.read_csv(results_dir / 'phase9_recommendations_validation.csv')
test_posthoc = pd.read_csv(results_dir / 'phase9_recommendations_test.csv')
validation_metrics = pd.read_csv(results_dir / 'phase9_metrics_validation.csv')
test_metrics = pd.read_csv(results_dir / 'phase9_metrics_test.csv')
validation_associations = pd.read_csv(results_dir / 'phase9_associations_validation.csv')
audit = json.loads((results_dir / 'phase9_audit.json').read_text(encoding='utf-8'))
print('Audit:', audit['overall_status'])

## Validation-derived recommendation method

MAINTAIN is used for Normal risk. CRITICAL_PROTECTIVE_ACTION follows Critical risk. INCREASE_COOLING is limited to Elevated risk with current_abs >= 20 A and flow below the validation P75 active-load reference. Other High/Elevated cases become HIGH_COOLING_ATTENTION. These are decision-support states, not causal control laws.

In [ ]:
config

## Validation results

Validation recommendation behavior is evaluated separately for ID16 and ID22 and both horizons. Recommendation transitions are counted without crossing run boundaries.

In [ ]:
validation_metrics

In [ ]:
validation_associations

In [ ]:
validation[['split', 'exp_id', 'time_s', 'horizon', 'liquid_flow', 'current_abs', 'predicted_future_temperature', 'risk_category', 'recommendation', 'recommendation_basis']].head()

## Diagnostic figures

Figures show validation cooling context, recommendation counts, and recommendation timelines.

In [ ]:
names = ['45_phase9_validation_flow_temperature.png', '46_phase9_recommendation_counts.png', '47_phase9_validation_recommendation_timeline.png']
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, name in zip(axes, names):
    ax.imshow(plt.imread(figures_dir / name)); ax.set_title(name); ax.axis('off')
plt.tight_layout()

## Held-out post-hoc evaluation

ID18 and ID19 are shown after validation rules were frozen. Their recommendation states were not used to tune thresholds, models, or logic.

In [ ]:
test_metrics

In [ ]:
test_posthoc[['split', 'exp_id', 'time_s', 'horizon', 'liquid_flow', 'current_abs', 'predicted_future_temperature', 'risk_category', 'recommendation']].head()

## Audit, limitations, and handoff

The audit checks unchanged Phase 0-8 artifacts, exact memberships, validation-only rules, no model retraining, temporal alignment, duplicate absence, and output completeness. Flow/temperature relationships are associations, not causal evidence. Phase 9 is ready for a future Phase 10 dashboard only after explicit authorization.

In [ ]:
assert audit['overall_status'] == 'PASS'
assert audit['methodology_source'] == 'validation runs ID16 and ID22 only'
assert audit['checks']['no_test_tuning']
assert audit['checks']['no_model_retraining']
audit['checks']